<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

# CP101: Lab 8 - Street Network Analysis & Isochrones

# Street Network Analysis with OSMnx

## What you'll learn

By the end of this tutorial, you'll be able to:
1. Download street network data from OpenStreetMap
2. Understand the basics of graph theory (nodes and edges)
3. Convert between graphs and GeoDataFrames
4. Calculate network statistics like density and centrality
5. Find shortest paths between locations
6. Create isochrone maps (travel time zones)

---

## Why network analysis?

Think about questions like:
- What's the fastest way to bike from BART to campus?
- How walkable is this neighborhood?
- Which streets carry the most traffic?

These are all network analysis questions. We'll use two main tools:
- **OSMnx** — downloads and analyzes street networks from OpenStreetMap
- **NetworkX** — the underlying Python library for graph analysis

If you use OSMnx in research, please cite:
> Boeing, G. 2017. "OSMnx: New Methods for Acquiring, Constructing, Analyzing, and Visualizing Complex Street Networks." *Computers, Environment and Urban Systems* 65, 126-139.


## Background: What Is an Isochrone?

An **isochrone** is a line (or area) that connects all points reachable from a single origin within the same amount of travel time or travel distance. The word comes from the Greek *isos* (equal) and *chronos* (time). You have seen the concept before on topographic maps, where contour lines connect points of equal elevation. An isochrone is the same idea applied to time or distance on a network.

Urban planners use isochrones to answer questions such as:

- How many residents can walk to a bus stop within 10 minutes?
- Which grocery stores fall inside a 15-minute cycling radius from a proposed housing site?
- Does the neighborhood meet the criteria of a "15-minute city"?

A key advantage of isochrones is that they **convert a routing problem into a spatial join problem**. Instead of computing individual routes between an origin and hundreds of destinations, you build the isochrone polygon once and then intersect it with a point layer of destinations. This is much faster at scale.

In this notebook we focus on **walking isochrones** built from OSM networks.

---

In [ ]:
# Initialize Otter
import otter
grader = otter.Notebook("CP101_LAB08.ipynb")

---

# Part 1: Setup

Let's get our libraries installed and imported.

In [ ]:
# Install all required packages
# geopandas with all optional dependencies. The [all] flag gives us extra goodies: folium for interactive maps,
# osmnx: street network analysis
# networkx: graph algorithms (usually comes with osmnx, but let's be safe)
# shapely: geometric operations (used for isochrones)

!uv pip install -q osmnx networkx "geopandas[all]"

In [ ]:
import osmnx as ox
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt

# Suppress warnings to keep output clean
import warnings
warnings.filterwarnings("ignore")

---

# Part 2: Downloading a street network

OSMnx makes this almost embarrassingly easy. Give it a place name, and it fetches the street network from OpenStreetMap.

In [ ]:
# Be specific with place names to avoid ambiguity.
# "Berkeley" could be Berkeley, CA or Berkeley, UK!

place_name = "Berkeley, CA, USA"

# You could also try:
# place_name = "Manhattan, New York, USA"
# place_name = "Downtown Oakland, CA, USA"

In [ ]:
# This queries OpenStreetMap and builds a graph. Takes 30-60 seconds.
#
# network_type options:
#   'all'   — everything (pedestrian paths, bike lanes, roads)
#   'drive' — car-accessible streets only
#   'walk'  — pedestrian-accessible paths
#   'bike'  — bike-accessible routes

G = ox.graph_from_place(place_name, network_type='all')

print(f"Got the street network for {place_name}!")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## What is actually in the walk network?

Before doing any analysis, we need to understand what data we are working with. `network_type='walk'` does not return a pedestrian-only network. It returns all edges where walking is not explicitly forbidden in OpenStreetMap.

The table below shows the main `highway` types you will find in a walk network and what each one actually represents.

| `highway` value | What it represents | Pedestrian infrastructure? |
|---|---|---|
| `footway` | Dedicated footpath, often alongside a road or through a park | Yes |
| `pedestrian` | Street or plaza designed primarily for pedestrian use | Yes |
| `path` | Informal or unpaved path, often in parks or natural areas | Yes |
| `steps` | Stairway | Yes |
| `residential` | Low-traffic street in a residential area | Road |
| `unclassified` | Minor road, below tertiary in the road hierarchy | Road |
| `tertiary` | Minor collector road | Road |
| `secondary` | Moderate-traffic arterial | Road |
| `primary` | Major arterial road | Road |
| `service` | Access road: parking lot, driveway, or alley | Road |
| `living_street` | Very low-speed street shared by pedestrians and vehicles | Road |

Roads are included because pedestrians are legally permitted to use them. Let us look at what actually appears in the Berkeley network.

In [ ]:
# Get a temporary edges GeoDataFrame for inspection only
# (We will re-derive edges from the projected graph in Part 4)
_, edges_raw = ox.graph_to_gdfs(G)
edges_raw = edges_raw.reset_index()

# highway can store a list when an edge carries multiple tags
# .explode() flattens those lists into individual rows before counting
highway_counts = (
    edges_raw['highway']
    .explode()
    .value_counts()
)
print(highway_counts)

As expected, `residential`, `secondary`, and other road types appear alongside pedestrian-specific types. Now let us see what is left if we keep only the pedestrian infrastructure.

In [ ]:
foot_types = ['footway', 'path', 'pedestrian', 'steps']

# .explode() handles edges where highway is stored as a list
# .groupby(level=0).any() collapses back to one True/False per original edge
mask_foot = (
    edges_raw['highway']
    .explode()
    .isin(foot_types)
    .groupby(level=0)
    .any()
)

edges_foot_only = edges_raw[mask_foot]

print(f"All walk edges:  {len(edges_raw)}")
print(f"Foot-only edges: {len(edges_foot_only)}")
pct = 100 * (1 - len(edges_foot_only) / len(edges_raw))
print(f"Removed:         {pct:.0f}% of edges")

In [ ]:
fig, axes = plt.subplots(
    1,
    2,
    figsize=(14, 6)
)

edges_raw.plot(
    ax=axes[0],
    linewidth=0.5,
    color='steelblue'
)
axes[0].set_title('Full walk network (including roads)')
axes[0].set_axis_off()

edges_foot_only.plot(
    ax=axes[1],
    linewidth=0.8,
    color='tomato'
)
axes[1].set_title('Foot-only infrastructure (footway, path, pedestrian, steps)')
axes[1].set_axis_off()

plt.suptitle('Berkeley: full walk network vs. foot-only infrastructure')
plt.tight_layout()
plt.show()

The foot-only network is sparse. Large areas have no footway edges at all, even though pedestrians clearly walk there. This reflects an OSM data completeness problem: sidewalks and dedicated footpaths are mapped far less consistently than roads.

If we tried to route on the foot-only network, the ego graph would fail for most origin points because the network is disconnected. For this reason we will use the full walk network throughout the lab. This is a known limitation of OSM data. Research tools such as Tile2Net are being developed to address this gap by detecting sidewalk infrastructure from aerial imagery.

---

# Part 3: A quick graph theory primer

Before we go further, let's talk about what we just downloaded.

## Graphs 101

A **graph** is just:
- **Nodes** (aka vertices): the dots — in our case, intersections
- **Edges** (aka links): the lines connecting them — street segments

## Directed vs undirected

In an **undirected** graph, edges work both ways (like a two-way street).

In a **directed** graph, edges have a direction (think one-way streets).

Street networks are **directed** because one-way streets exist. They're also "multi" graphs because you can have multiple edges between the same two nodes (parallel roads, different travel modes, etc.).



In [ ]:
# Let's confirm: OSMnx gives us a MultiDiGraph
# Multi = multiple edges between same nodes allowed
# Di = directed (edges have direction)

print(f"Graph type: {type(G)}")

---

# Part 4: Projection

The raw data is in lat/long (WGS84). That's fine for showing on a map, but terrible for measuring distances — a degree of latitude isn't the same length everywhere on Earth.

For analysis, we project to a local coordinate system (UTM) where units are in meters.

In [ ]:
# OSMnx figures out the right UTM zone automatically. Nice.

G_proj = ox.project_graph(G)

print("Projected to UTM (meters). Now distances will be accurate.")

---

# Part 5: Graphs ↔ GeoDataFrames

Graphs are great for network analysis, but sometimes you want a good old GeoDataFrame. OSMnx makes conversion easy.

In [ ]:
# This returns a tuple: (nodes_gdf, edges_gdf)
# Nodes are Points (intersections), edges are LineStrings (streets)

nodes_proj, edges_proj = ox.graph_to_gdfs(G_proj)
nodes_deg, edges_deg = ox.graph_to_gdfs(G)  # unprojected version too

print(f"Nodes: {len(nodes_proj):,} intersections")
print(f"Edges: {len(edges_proj):,} street segments")

In [ ]:
# Quick sanity check on geometry types and CRS

print("NODES")
print(f"  Geometry: {nodes_proj.geometry.type.unique()}")
print(f"  CRS: {nodes_proj.crs}")

print("\nEDGES")
print(f"  Geometry: {edges_proj.geometry.type.unique()}")
print(f"  CRS: {edges_proj.crs}")

In [ ]:
# You can also grab just nodes or just edges

nodes_only = ox.graph_to_gdfs(G_proj, edges=False)
edges_only = ox.graph_to_gdfs(G_proj, nodes=False)

print(f"Got {len(nodes_only)} nodes and {len(edges_only)} edges separately")

In [ ]:
# Let's visualize projected vs unprojected side by side
# Notice the subtle shape difference — that's the projection at work

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

edges_proj.plot(ax=axes[0], linewidth=0.5, color='steelblue')
axes[0].set_title("Projected (UTM, meters)")
axes[0].set_xlabel("Easting (m)")
axes[0].set_ylabel("Northing (m)")

edges_deg.plot(ax=axes[1], linewidth=0.5, color='forestgreen')
axes[1].set_title("Unprojected (WGS84, degrees)")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")

plt.tight_layout()
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Try it Out: how long is Telegraph Avenue?

Now that the network is in meters, we can measure real streets. How long is Telegraph Avenue in Berkeley?

The cell below starts by making `edges_undirected`, the edges of an **undirected** copy of `G_proj`. A two-way street is stored as two edges in our directed graph, one for each direction, so the undirected copy keeps each street segment only once.

1. Filter `edges_undirected` to the edges whose `name` is `'Telegraph Avenue'` and assign the result to `telegraph`.
2. Assign the total length of those edges in **kilometers** to `telegraph_km`. The `length` column is in meters.

*Hint:* filtering works the same way as in a regular DataFrame: `gdf[gdf['column'] == value]`.

In [ ]:
# Each street segment once, not once per direction
edges_undirected = ox.graph_to_gdfs(ox.convert.to_undirected(G_proj), nodes=False)

# Filter the edges to only include Telegraph Avenue
telegraph = ...

# Calculate the total length of Telegraph Avenue in kilometers
telegraph_km = ...

print(f"Telegraph Avenue: {telegraph_km:.2f} km")

In [ ]:
grader.check("q3")

---

# Part 6: Spatial filtering

GeoPandas has a handy `.cx[]` indexer that filters by bounding box. Super useful for zooming into an area of interest.

In [ ]:
# Let's filter to just the downtown Berkeley / campus area
# Syntax: gdf.cx[min_lon:max_lon, min_lat:max_lat]

min_lat, max_lat = 37.866, 37.876
min_lon, max_lon = -122.271, -122.247

edges_filtered = edges_deg.cx[min_lon:max_lon, min_lat:max_lat]

print(f"Full network: {len(edges_deg):,} edges")
print(f"Filtered: {len(edges_filtered):,} edges")

- `.cx[min_lon:max_lon, min_lat:max_lat]` is a spatial indexer built into GeoPandas.
It filters a GeoDataFrame to only the rows whose geometry falls within the bounding box you specify. The coordinates are in degrees (longitude, latitude) because

- `edges_deg` is in WGS 84. This is much faster than a full spatial join — it is just a bounding box clip.

In [ ]:
# Compare the full network with our filtered subset

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

edges_deg.plot(ax=axes[0], linewidth=0.3, color='gray')
axes[0].set_title("Full Berkeley network")

edges_filtered.plot(ax=axes[1], linewidth=0.5, color='crimson')
axes[1].set_title("Filtered to downtown/campus")

plt.tight_layout()
plt.show()

In [ ]:
edges_filtered

---

# Part 7: Network statistics

Now for the fun stuff. Let's measure some properties of our network.

In [ ]:
# First, we need the area covered by our network.
# We'll take the convex hull of all nodes (the smallest polygon containing them all)

network_hull = nodes_proj.union_all().convex_hull
area_m2 = network_hull.area
area_km2 = area_m2 / 1_000_000

print(f"Network covers {area_km2:.1f} km²")

In [ ]:
# OSMnx calculates a bunch of useful stats for us

stats = ox.basic_stats(G_proj, area=area_m2, clean_int_tol=15)

print("NETWORK STATS")
print(f"  Nodes (intersections): {stats['n']:,}")
print(f"  Edges (street segments): {stats['m']:,}")
print(f"  Avg connections per intersection: {stats['k_avg']:.1f}")
print(f"  Total street length: {stats['edge_length_total']/1000:.0f} km")
print(f"  Avg edge length: {stats['edge_length_avg']:.0f} m")
print(f"  Street density: {stats['street_density_km']:.0f} m/km²")
print(f"  Circuity: {stats['circuity_avg']:.2f}")
print("    (1.0 = perfectly straight routes, higher = more winding)")

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Quick glossary

| Metric | What it means | Why it matters |
|--------|--------------|----------------|
| **Density** | Street length per area | More = more connected |
| **Centrality** | How "important" a street is | Identifies key corridors |
| **Circuity** | Actual route ÷ straight line | Measures grid efficiency |
| **Connectivity** | Ways to get between points | Network resilience |

In [ ]:
# OSMnx's plot_graph gives us a nice visualization
# Black background helps the network pop

fig, ax = ox.plot_graph(G, figsize=(12, 12))
plt.show()

---

# Part 8: Centrality analysis

Which streets are most central to the network? Closeness centrality measures how "close" each edge is to all other edges. High centrality = important corridor.

In [ ]:
# This is a bit slow (1-2 min) because we're computing centrality for every edge.
# The trick: convert to a "line graph" where edges become nodes, then use standard node centrality algorithms.

# Centrality on the full graph is slow — we use the filtered area instead
# Keep only nodes that appear in edges_filtered

print("Computing edge centrality... (grab a coffee)")

nodes_in_area = set(edges_filtered.index.get_level_values('u')) | set(edges_filtered.index.get_level_values('v'))
G_filtered = G.subgraph(nodes_in_area)
line_graph = nx.line_graph(G_filtered)
edge_centrality = nx.closeness_centrality(line_graph)
nx.set_edge_attributes(G_filtered, edge_centrality, 'edge_centrality')


print(f"Full graph:     {len(G.nodes):,} nodes")
print(f"Filtered graph: {len(G_filtered.nodes):,} nodes")

In [ ]:
# Now let's visualize it. Yellow = high centrality, dark = low.
# Closeness Centrality (yellow = most central)

ec = ox.plot.get_edge_colors_by_attr(G, 'edge_centrality', cmap='inferno')

fig, ax = ox.plot_graph(
    G_filtered,
    edge_color=ec,
    edge_linewidth=2,
    node_size=0,
    figsize=(12, 12)
)



<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Try it Out: which street is the most central?

The map above shows the most central streets in yellow. Which street is the most central one? Answer it with data.

Centrality is calculated for each **edge**, and a street is made of many edges. Also, OSMnx merges connected segments into a single edge, so some edges have a *list* of names (for example `['Sproul Plaza', 'Sather Road']`). To rank whole streets, give each street the average centrality of all its edges.

1. Convert `G_filtered` to an edges GeoDataFrame and assign it to `edges_central`. Each edge has an `edge_centrality` column.
2. Use `.explode('name')` so each street name gets its own row, then group by `name` and take the mean `edge_centrality`. Sort from highest to lowest and assign the result to `street_centrality`.
3. Assign the name of the most central street to `most_central_street`.

*Hints:* `ox.graph_to_gdfs(..., nodes=False)` returns just the edges (see Part 5). You used `.explode()` on the `highway` column in Part 2. `sort_values(ascending=False)` puts the largest values first, and `.index[0]` gives the first label of a Series.

In [ ]:

# Convert the filtered graph to a GeoDataFrame of edges
edges_central = ...

# Average edge centrality for each street name, highest first
street_centrality = ...

# The most central street
most_central_street = ...

print(f"Most central street: {most_central_street}")
street_centrality.head(10)

In [ ]:
grader.check("q2")

**Discussion questions:**

1. Look at the top 10 in `street_centrality`. Is your most central street a major street or a short one? Does the ranking match your sense of downtown Berkeley and campus?
2. We gave each street the *average* centrality of its edges. What would change if we used the *total* (sum) instead?

*Type your answers here.*

---

# Part 9: Routing

The classic network analysis question: what's the best route from A to B?

In [ ]:
# Our origin and destination
# Tip: I got these coords from OpenStreetMap (right-click → "Show address")

# Berkeley BART station
origin_lat, origin_lon = 37.870345, -122.26776

# Wurster Hall (CED building on campus)
dest_lat, dest_lon = 37.870778, -122.254709

print(f"From: Berkeley BART ({origin_lat}, {origin_lon})")
print(f"To: Wurster Hall ({dest_lat}, {dest_lon})")

In [ ]:
# Our points probably aren't exactly on a node, so we find the nearest ones
# Note the X=lon, Y=lat order — easy to mix up!

origin_node = ox.distance.nearest_nodes(G, X=origin_lon, Y=origin_lat)
dest_node = ox.distance.nearest_nodes(G, X=dest_lon, Y=dest_lat)

print(f"Origin node: {origin_node}")
print(f"Destination node: {dest_node}")

In [ ]:
# For routing, we need travel times on edges.
# Let's get a walking network and add speed/time attributes.

G_walk = ox.graph_from_place(place_name, network_type='walk')
G_walk = ox.add_edge_speeds(G_walk)
G_walk = ox.add_edge_travel_times(G_walk)

print("Walking network ready with travel times")

In [ ]:
# Let's peek at what attributes edges have now

edges_walk = ox.graph_to_gdfs(G_walk, nodes=False)

print("Edge columns:", edges_walk.columns.tolist())
print("\nSample data:")
edges_walk[['name', 'length', 'speed_kph', 'travel_time']].head(3)

In [ ]:
# Find the shortest path by travel time
# Under the hood, this uses Dijkstra's algorithm

route = ox.shortest_path(G_walk, origin_node, dest_node, weight='travel_time')

print(f"Route has {len(route)} nodes")
print(f"First few: {route[:5]}")
print(f"Last few: {route[-5:]}")

In [ ]:
# Each node ID gives us access to its attributes

mid_node = route[len(route)//2]
print(f"Middle node: {mid_node}")
print(f"Attributes: {G_walk.nodes[mid_node]}")

In [ ]:
# Static plot of the route: "BART to Wurster Hall (fastest walking route)"

fig, ax = ox.plot_graph_route(
    G_walk, route,
    route_linewidth=6,
    route_color='red',
    node_size=0,
    figsize=(12, 12)
)




In [ ]:
# Interactive version — you can zoom and pan

route_gdf = ox.routing.route_to_gdf(G_walk, route)
route_gdf.explore(
    tiles="openstreetmap",
    style_kwds={"weight": 5, "color": "red"}
    )

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## ✏️ Try it Out: plan your own walk

You're at Berkeley BART and want to walk to North Berkeley BART (latitude 37.8740, longitude −122.2834) instead of taking the train one stop. Use the same steps as the route above.

1. Find the nearest node in `G_walk` to North Berkeley BART and assign it to `north_node`.
2. Find the shortest walking route **by distance** from `origin_node` to `north_node` and assign it to `route_north`.
3. Assign the route's total length in meters to `route_north_m`.
4. Assuming a 4.5 km/h walking pace, assign the walk time in minutes to `route_north_min`.

*Hints:* `nearest_nodes` takes `X=` longitude and `Y=` latitude. `ox.routing.route_to_gdf()` turns a route into a GeoDataFrame with a `length` column in meters.

In [ ]:
# Walking speed of 4.5 km/hin meters per minute 
meters_per_min = 4.5 * 1000 / 60   # 75 m/min

# 1. Nearest walk-network node to North Berkeley BART (X = longitude, Y = latitude)
north_node = ...

# 2. Shortest walking route by distance
route_north = ...

# 3. Total route length in meters
route_north_m = ...

# 4. Walk time in minutes at 4.5 km/h
route_north_min = ...


print(f"Route length: {route_north_m:.0f} m")
print(f"Walk time: {route_north_min:.1f} min")

In [ ]:
grader.check("q1")

### Is "shortest distance" the same as "shortest time"?

Let's compare.


In [ ]:
# Constant speed: every edge is walked at the same pace, so time = length / speed
walk_speed_mps = 4.5 * 1000 / 3600  # 4.5 km/hr in meters per second
for u, v, k, data in G_walk.edges(keys=True, data=True):
    data['travel_time_const'] = data['length'] / walk_speed_mps

route_by_time = ox.shortest_path(G_walk, origin_node, dest_node, weight='travel_time_const')
route_by_dist = ox.shortest_path(G_walk, origin_node, dest_node, weight='length')

_, edges_walk = ox.graph_to_gdfs(G_walk)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='k')

for ax in axes:
    edges_walk.plot(
        ax=ax,
        linewidth=0.5,
        color='#999999',
        alpha=0.5
    )
    ax.set_facecolor('k')
    ax.set_axis_off()

ox.plot_graph_route(
    G_walk,
    route_by_time,
    ax=axes[0],
    route_linewidth=4,
    node_size=0,
    route_color='steelblue',
    show=False,
    close=False,
    bgcolor='k'
)
axes[0].set_title("Shortest TIME", color='white')

ox.plot_graph_route(
    G_walk,
    route_by_dist,
    ax=axes[1],
    route_linewidth=4,
    node_size=0,
    route_color='limegreen',
    show=False,
    close=False,
    bgcolor='k'
)
axes[1].set_title("Shortest DISTANCE", color='white')

# Zoom in on the route area
route_nodes = set(route_by_time) | set(route_by_dist)
xs = [G_walk.nodes[n]['x'] for n in route_nodes]
ys = [G_walk.nodes[n]['y'] for n in route_nodes]
for ax in axes:
    ax.set_xlim(min(xs) - 0.002, max(xs) + 0.002)
    ax.set_ylim(min(ys) - 0.004, max(ys) + 0.004)

plt.tight_layout()
plt.show()

On a walk network every edge gets the same speed, so travel_time = length / constant — minimizing time and minimizing distance are the same problem.
On a drive network, speed varies by road type, so a longer route on a fast arterial can beat a shorter route on slow residential streets.
This is why the two routes look identical here: try it on a drive network to see them diverge.

In [ ]:
# Variable speed: travel_time comes from add_edge_speeds (speed depends on the road)
route_by_time = ox.shortest_path(G_walk, origin_node, dest_node, weight='travel_time')
route_by_dist = ox.shortest_path(G_walk, origin_node, dest_node, weight='length')

_, edges_walk = ox.graph_to_gdfs(G_walk)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), facecolor='k')

for ax in axes:
    edges_walk.plot(
        ax=ax,
        linewidth=0.5,
        color='#999999',
        alpha=0.5
    )
    ax.set_facecolor('k')
    ax.set_axis_off()

ox.plot_graph_route(
    G_walk,
    route_by_time,
    ax=axes[0],
    route_linewidth=4,
    node_size=0,
    route_color='steelblue',
    show=False,
    close=False,
    bgcolor='k'
)
axes[0].set_title("Shortest TIME", color='white')

ox.plot_graph_route(
    G_walk,
    route_by_dist,
    ax=axes[1],
    route_linewidth=4,
    node_size=0,
    route_color='limegreen',
    show=False,
    close=False,
    bgcolor='k'
)
axes[1].set_title("Shortest DISTANCE", color='white')

# Zoom in on the route area
route_nodes = set(route_by_time) | set(route_by_dist)
xs = [G_walk.nodes[n]['x'] for n in route_nodes]
ys = [G_walk.nodes[n]['y'] for n in route_nodes]
for ax in axes:
    ax.set_xlim(min(xs) - 0.002, max(xs) + 0.002)
    ax.set_ylim(min(ys) - 0.004, max(ys) + 0.004)

plt.tight_layout()
plt.show()

---

# Part 10: Isochrone maps

An isochrone shows everywhere you can reach within X minutes from a starting point. Super useful for accessibility analysis.

In [ ]:
from shapely.geometry import Point

# We'll make isochrones for 5, 10, 15, 20, and 25 minutes walks
trip_times = [5, 10, 15, 20, 25]  # in minutes
walk_speed = 4.5  # walking speed in km/hour

center_node = origin_node  # starting from BART

print(f"Creating isochrones for {trip_times} minute walks at {walk_speed} km/hr")

In [ ]:
# We need to add a 'time' attribute to each edge (in minutes)
# Formula: time = distance / speed

G_walk_proj = ox.project_graph(G_walk)

# Convert km/hr to m/min: 4 km/hr = 4000 m / 60 min = 66.7 m/min
meters_per_min = walk_speed * 1000 / 60
print(f"That's {meters_per_min:.1f} meters per minute")

# Add time to each edge
for u, v, k, data in G_walk_proj.edges(data=True, keys=True):
    data['time'] = data['length'] / meters_per_min

print("Added 'time' attribute to all edges")

In [ ]:
# For each time threshold, find all reachable nodes and make a polygon
# ego_graph() gives us all nodes within a given "radius" (here, time in minutes)

isochrone_polys = []

for minutes in sorted(trip_times, reverse=True):  # largest first for proper layering
    # Get all nodes reachable within this many minutes
    subgraph = nx.ego_graph(G_walk_proj, center_node, radius=minutes, distance='time')

    # Convert nodes to points
    points = [Point(data['x'], data['y']) for node, data in subgraph.nodes(data=True)]

    # Wrap them in a convex hull
    hull = gpd.GeoSeries(points).union_all().convex_hull
    isochrone_polys.append(hull)

    print(f"{minutes} min: {len(points)} nodes reachable")

In [ ]:
# Package into a GeoDataFrame

crs = ox.graph_to_gdfs(G_walk_proj)[0].crs

isochrones = gpd.GeoDataFrame(
    {'minutes': sorted(trip_times, reverse=True), 'geometry': isochrone_polys},
    crs=crs
)

isochrones

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Plots nodes you can reach on foot within each time

How far can you walk in 5, 10, 15, 20, and 25 minutes from the origin node? We'll use NetworkX to induce a subgraph of G within each distance, based on trip time and travel speed.

In [ ]:
iso_colors = ox.plot.get_colors(n=len(trip_times), cmap="plasma", start=0)

In [ ]:
# color the nodes according to isochrone then plot the street network
node_colors = {}
for trip_time, color in zip(sorted(trip_times, reverse=True), iso_colors, strict=False):
    subgraph = nx.ego_graph(G, center_node, radius=trip_time, distance="time")
    for node in subgraph.nodes():
        node_colors[node] = color
nc = []
ns = []
for node in G_walk.nodes():
    nc.append(node_colors.get(node, "none"))
    if node in node_colors:
        ns.append(15)
    else:
        ns.append(0)
fig, ax = ox.plot.plot_graph(
    G_walk,
    node_color=nc,
    node_size=ns,
    node_alpha=0.8,
    edge_linewidth=0.2,
    edge_color="#999999",

)

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Plot the time-distances as isochrones

How far can you walk in 5, 10, 15, 20, and 25 minutes from the origin node? We'll use a convex hull, which isn't perfectly accurate. A concave hull would be better, but shapely doesn't offer that.

In [ ]:
# make the isochrone polygons
isochrone_polys = []
for trip_time in sorted(trip_times, reverse=True):
    subgraph = nx.ego_graph(
        G_walk,
        center_node,
        radius=trip_time,
        distance="time"
        )

    node_points = [Point((data["x"], data["y"])) for node, data in subgraph.nodes(data=True)]
    bounding_poly = gpd.GeoSeries(node_points).union_all().convex_hull
    isochrone_polys.append(bounding_poly)
gdf = gpd.GeoDataFrame(geometry=isochrone_polys)

In [ ]:
# plot the network then add isochrones as colored polygon patches
fig, ax = ox.plot.plot_graph(
    G_walk,
    show=False,
    close=False,
    edge_color="#999999",
    edge_alpha=0.2,
    node_size=0,
)
gdf.plot(ax=ax, color=iso_colors, ec="none", alpha=0.6, zorder=-1)
plt.show()

<hr style="border: 5px solid #003262;" />
<hr style="border: 1px solid #fdb515;" />

## Or, plot isochrones as buffers to get more faithful isochrones than convex hulls can offer

In [ ]:
from shapely.geometry import Polygon, LineString, MultiPolygon

from shapely.geometry import Point, Polygon, LineString, MultiPolygon
import geopandas as gpd
import networkx as nx
import matplotlib.pyplot as plt
import osmnx as ox


def make_iso_polys(
    G,
    center_node,
    trip_times,
    edge_buff=25,
    node_buff=50,
    infill=False,
):
    """
    Build isochrone polygons from a projected graph.

    Parameters
    ----------
    G : nx.MultiDiGraph
        Projected graph.
    center_node : int
        Center node in the same graph.
    trip_times : list[int | float]
        Travel time thresholds.
    edge_buff : float
        Edge buffer in graph CRS units, usually meters.
    node_buff : float
        Node buffer in graph CRS units, usually meters.
    infill : bool
        If True, remove holes from polygons.

    Returns
    -------
    list
        List of Shapely geometries.
    """
    isochrone_polys = []

    for trip_time in sorted(trip_times, reverse=True):
        subgraph = nx.ego_graph(
            G,
            center_node,
            radius=trip_time,
            distance="time",
        )

        node_points = [
            Point(
                (
                    data["x"],
                    data["y"],
                )
            )
            for _, data in subgraph.nodes(data=True)
        ]

        nodes_gdf = gpd.GeoDataFrame(
            {"id": list(subgraph.nodes)},
            geometry=node_points,
            crs=G.graph["crs"],
        ).set_index("id")

        edge_lines = []
        for u, v, k, data in subgraph.edges(keys=True, data=True):
            if "geometry" in data:
                edge_lines.append(data["geometry"])
            else:
                f = nodes_gdf.loc[u].geometry
                t = nodes_gdf.loc[v].geometry
                edge_lines.append(
                    LineString(
                        [
                            f,
                            t,
                        ]
                    )
                )

        node_buffers = []
        if node_buff > 0:
            node_buffers = list(nodes_gdf.geometry.buffer(node_buff))

        edge_buffers = list(
            gpd.GeoSeries(
                edge_lines,
                crs=G.graph["crs"],
            ).buffer(edge_buff)
        )

        all_geoms = node_buffers + edge_buffers
        new_iso = gpd.GeoSeries(
            all_geoms,
            crs=G.graph["crs"],
        ).union_all()

        if infill:
            if new_iso.geom_type == "Polygon":
                new_iso = Polygon(new_iso.exterior)
            elif new_iso.geom_type == "MultiPolygon":
                exterior_polys = []
                for poly in new_iso.geoms:
                    exterior_polys.append(Polygon(poly.exterior))
                new_iso = MultiPolygon(exterior_polys)

        isochrone_polys.append(new_iso)

    return isochrone_polys


G_proj = ox.project_graph(G_walk)

center_node_proj = ox.distance.nearest_nodes(
    G_proj,
    X=G_proj.nodes[center_node]["x"],
    Y=G_proj.nodes[center_node]["y"],
)

isochrone_polys = make_iso_polys(
    G=G_proj,
    center_node=center_node_proj,
    trip_times=trip_times,
    edge_buff=25,
    node_buff=0,
    infill=True,
)

gdf = gpd.GeoDataFrame(
    geometry=isochrone_polys,
    crs=G_proj.graph["crs"],
).to_crs("EPSG:4326")

fig, ax = ox.plot_graph(
    G_walk,
    show=False,
    close=False,
    edge_color="#999999",
    edge_alpha=0.2,
    node_size=0,
)

gdf.plot(
    ax=ax,
    color=iso_colors,
    edgecolor="none",
    alpha=0.6,
    zorder=1,
)

plt.show()

In [ ]:
# Interactive map! Darker = longer walk.

isochrones_wgs = isochrones.to_crs(epsg=4326)

isochrones_wgs.explore(
    column='minutes',
    cmap='Reds_r',
    tiles='openstreetmap',
    legend=True,
    style_kwds={'fillOpacity': 0.5}
)

---

# Recap

We covered a lot! Here's what you can now do:

| Task | Function |
|------|----------|
| Download a network | `ox.graph_from_place()` |
| Project to meters | `ox.project_graph()` |
| Convert to GeoDataFrame | `ox.graph_to_gdfs()` |
| Get network stats | `ox.basic_stats()` |
| Find shortest path | `ox.shortest_path()` |
| Make isochrones | `nx.ego_graph()` + convex hull |

## Ideas to try

- Compare networks: `'drive'` vs `'walk'` vs `'bike'`
- Analyze your own neighborhood
- Compare circuity across different cities
- Build an accessibility analysis with isochrones

## Resources

- [OSMnx docs](https://osmnx.readthedocs.io/)
- [NetworkX docs](https://networkx.org/documentation/stable/)
- [Geoff Boeing's examples](https://github.com/gboeing/osmnx-examples)

## Submission

Make sure you have run all cells in your notebook in order before running the cell below, so that all images/graphs appear in the output. The cell below will generate a zip file for you to submit.

Make sure you have saved your notebook before running this cell. Then run this cell and click on the download link to get your submission zip.

In [ ]:
grader.export(pdf=False, force_save=True, run_tests=True)